# Active-learning benchmark results

Plots from `al_benchmark/results.csv`. See [`../docs/AL_Benchmark.md`](../docs/AL_Benchmark.md) for the methodology and conclusions.

The benchmark replays four MEL-budget allocation policies against the SRG-score oracle at three budgets (1K / 5K / 25K synthon-equivalents) with five seeds. Lower regret and higher top-K recall are better.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
from paths import PROJECT_ROOT

df = pd.read_csv(PROJECT_ROOT / 'al_benchmark' / 'results.csv')
df.head()

In [ ]:
summary = df.groupby(['policy', 'budget_total']).agg(
    regret_mean=('regret_sum_topB', 'mean'),
    regret_sem=('regret_sum_topB', 'sem'),
    recall50_mean=('recall_top50', 'mean'),
    recall50_sem=('recall_top50', 'sem'),
    hits_mean=('n_hits', 'mean'),
    hits_sem=('n_hits', 'sem'),
).reset_index()
summary

In [ ]:
# Regret vs budget (lower is better) — one line per policy.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
for ax, (metric, ylabel, lower_better) in zip(axes, [
    ('regret_sum_topB', 'Sum-top-B regret (lower = better)', True),
    ('recall_top50',    'Top-50 recall (higher = better)',    False),
    ('n_hits',          'Hits found (higher = better)',       False),
]):
    for policy, sub in df.groupby('policy'):
        agg = sub.groupby('budget_total')[metric].agg(['mean', 'sem']).reset_index()
        ax.errorbar(agg['budget_total'], agg['mean'], yerr=agg['sem'],
                    marker='o', capsize=3, label=policy)
    ax.set_xscale('log')
    ax.set_xlabel('Budget (synthon-equivalents)')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.suptitle('Policy comparison on the 7-MEL CB2_5ZTY oracle (5 seeds)')
fig.tight_layout()
plt.show()

## Reading the result

Numbers from the most recent run (see summary table above):

| Budget | baseline regret | greedy regret | TS regret | UCB regret |
|---|---|---|---|---|
| 1K   | ~22.2K | ~20.1K | ~20.4K | ~20.4K |
| 5K   | ~57.0K | ~86.7K (worse) | ~55.7K | ~56.8K |
| 25K  | ~160K  | ~288K (much worse) | ~149K | ~150K |

**TS and UCB are slightly but consistently better than the baseline at every budget; greedy is dramatically worse at high budgets** (it over-commits to the apparent best MEL early and forgoes hits hiding in the other MELs' tails).

Caveat: the oracle covers only 7 MELs (4 of which have ≥10K synthons). The conclusion is suggestive, not decisive — see the writeup's "What we don't know yet" section.